## Problem Formulation

**Domain:** Startup Analytics
**Problem Statement:**  Early identification of factors that influence startup success or failure.

Startup outcomes are influenced not only by internal financial and operational factors (e.g., funding, revenue, industry), but also by broader market trends and public interest.

**Main Question:**

- What factors most strongly predict whether a startup will succeed or fail?
- How accurately can startup profitability be predicted using structured financial and categorical features?
- Does external market interest, measured through Google Trends domain popularity, provide additional insight into startup profitability across regions?

## Data Collection Plan


To address the research questions, multiple complementary data sources are required. These sources combine structured historical startup data with externally collected indicators of market interest.


### Plan Table
- Two datasets are obtained from Kaggle and provide structured startup level information.
- One custom dataset is generated programmatically to aggregate industry level profitability signals.
- One additional dataset will be collected through web scraping using the Google Trends API (pytrends) to capture external market interest over time and across regions.



| **Data Source**                                                     | **Type of Data**                                                                                              | **Collection Mode**                        | **Format** | **Sampling** | **Storage**   | **Description / Purpose**                                                                                                                                                                                           |
| ------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------- | ------------------------------------------ | ---------- | ------------ | ------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Startup Failure Prediction Dataset (Kaggle)**                     | Quantitative (Funding Amount, Revenue, Burn Rate) + Qualitative (Market Size, Business Model, Startup Status) | Downloaded                                 | `.csv`     | No           | Local storage | Used to analyze determinants of startup success or failure and to derive profitability labels based on financial indicators.                                                                                        |
| **Startup Growth and Funding Trends (Kaggle)**                      | Quantitative (Funding Amount, Valuation, Revenue, Market Share) + Qualitative (Industry, Region, Exit Status) | Downloaded                                 | `.csv`     | No           | Local storage | Used to analyze growth, funding patterns, and profitability trends across industries and regions.                                                                                                                   |
| **Industry-Level Profitability Summary (Custom / Derived Dataset)** | Quantitative (Average Funding) + Binary Indicator (Industry Profitability)                                    | Programmatically generated from raw data   | `.csv`     | No           | Local storage | Aggregates startup-level data at the industry level. An industry is labeled as profitable if more than 50% of startups within it are profitable. Serves as a simplified, high-level signal for downstream analysis. |
| **Market Interest Trends (Google Trends – Planned)**                | Quantitative (Normalized search interest over time and by region)                                             | Scraped via Google Trends API (`pytrends`) | `.csv`     | No           | Local storage | Captures external market interest for startup-related domains (e.g., FinTech, HealthTech, AI). Used to contextualize profitability and growth patterns with public attention and trend momentum.                    |


In [15]:
import pandas as pd
import os

cwd = os.getcwd()

print("Current working directory:", cwd)

data_path = os.path.abspath(
    os.path.join(cwd, "..", "..", "app", "data", "startup_data_growth.csv")
)
data = pd.read_csv(data_path)

Current working directory: c:\Users\mraic\OneDrive\Desktop\Scoala\DS\1st_sem\Data_Collection_and_Modeling\Startup_Success\backend\ml\data_analysis


In [13]:
data

,Startup Name,Industry,Funding Rounds,Funding Amount (M USD),Valuation (M USD),Revenue (M USD),Employees,Market Share (%),Profitable,Year Founded,Region,Exit Status
0,Startup_1,IoT,1,101.09,844.75,67.87,1468,5.20,0,2006,Europe,Private
1,Startup_2,EdTech,1,247.62,3310.83,75.65,3280,8.10,1,2003,South America,Private
2,Startup_3,EdTech,1,109.24,1059.37,84.21,4933,2.61,1,1995,South America,Private
3,Startup_4,Gaming,5,10.75,101.90,47.08,1059,2.53,0,2003,South America,Private
4,Startup_5,IoT,4,249.28,850.11,50.25,1905,4.09,0,1997,Europe,Acquired
...,...,...,...,...,...,...,...,...,...,...,...,...
495,Startup_496,EdTech,2,181.86,2378.65,59.64,3331,0.58,1,1993,Europe,Private
496,Startup_497,AI,2,107.34,1394.58,10.22,2223,5.85,0,2019,South America,Private
497,Startup_498,E-Commerce,1,160.29,502.09,84.73,2222,4.32,0,2019,Australia,Private
498,Startup_499,Gaming,5,234.65,2814.52,53.16,4972,5.53,0,2011,Europe,Private


### The file that i created:

In [4]:
import os
import pandas as pd


def create_industry_funding_profitability_summary(
    input_csv_path: str,
    output_csv_path: str,
    industry_col: str = "Industry",
    funding_col: str = "Funding Amount (M USD)",
    profitable_col: str = "Profitable",
) -> pd.DataFrame:
    """
    Create a summary CSV with columns:
      - industry
      - funding (average funding per industry)
      - profitable (1 if >50% of startups in the industry are profitable, else 0)
    """
    if not os.path.exists(input_csv_path):
        raise FileNotFoundError(f"Input file not found: {input_csv_path}")

    df = pd.read_csv(input_csv_path)

    required = {industry_col, funding_col, profitable_col}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in input CSV: {sorted(missing)}")

    # Clean + coerce numeric columns safely
    df = df.copy()
    df[industry_col] = df[industry_col].astype(str).str.strip()

    # Funding may come as string; coerce to numeric
    df[funding_col] = pd.to_numeric(df[funding_col], errors="coerce")

    # Profitable should be 0/1; coerce and drop bad rows
    df[profitable_col] = pd.to_numeric(df[profitable_col], errors="coerce")

    # Drop rows missing key fields
    df = df.dropna(subset=[industry_col, funding_col, profitable_col])

    # Ensure profitable is treated as 0/1
    df[profitable_col] = df[profitable_col].astype(int)
    df = df[df[profitable_col].isin([0, 1])]

    grouped = df.groupby(industry_col, dropna=False).agg(
        funding_avg=(funding_col, "mean"),
        profitable_rate=(profitable_col, "mean"),  # mean of 0/1 = share profitable
    )

    summary = grouped.reset_index()
    summary["profitable"] = (summary["profitable_rate"] > 0.5).astype(int)

    # Output column names exactly as requested
    summary = summary.rename(columns={industry_col: "industry"})
    summary["funding"] = summary["funding_avg"]

    summary = summary[["industry", "funding", "profitable"]].sort_values(
        by=["industry"], ascending=True
    )

    # Ensure output folder exists
    out_dir = os.path.dirname(os.path.abspath(output_csv_path))
    if out_dir and not os.path.exists(out_dir):
        os.makedirs(out_dir, exist_ok=True)

    summary.to_csv(output_csv_path, index=False)

    return summary


In [11]:
import os

# Get current working directory (notebook location)
cwd = os.getcwd()

print("Current working directory:", cwd)

# Build path to the CSV
data_path = os.path.abspath(
    os.path.join(cwd, "..", "..", "app", "data", "startup_data_growth.csv")
)

output_path = os.path.abspath(
    os.path.join(cwd, "..", "..", "app", "data", "industry_funding_profitability_summary.csv")
)

print("Input CSV:", data_path)
print("Output CSV:", output_path)

summary_df = create_industry_funding_profitability_summary(
    input_csv_path=data_path,
    output_csv_path=output_path,
)

summary_df.head()


Current working directory: c:\Users\mraic\OneDrive\Desktop\Scoala\DS\1st_sem\Data_Collection_and_Modeling\Startup_Success\backend\ml\data_analysis
Input CSV: c:\Users\mraic\OneDrive\Desktop\Scoala\DS\1st_sem\Data_Collection_and_Modeling\Startup_Success\backend\app\data\startup_data_growth.csv
Output CSV: c:\Users\mraic\OneDrive\Desktop\Scoala\DS\1st_sem\Data_Collection_and_Modeling\Startup_Success\backend\app\data\industry_funding_profitability_summary.csv


,industry,funding,profitable
0,AI,138.339516,0
1,Cybersecurity,156.319804,0
2,E-Commerce,164.633000,1
3,EdTech,151.515270,0
4,FinTech,164.034225,0
